# Free-Fall — a quantitative teardown 🔬
### The carry-vs-crash decomposition · the −4.8 skew · the −83% day · the post-2018 resumption

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Naive short-vol survives?: Busted](https://img.shields.io/badge/Naive_short--vol_survives%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). A real carry with a ruinous tail.

> ⚠️ **Not investment advice.** SVXY vs SPY, daily total return (Yahoo), 2018–2026. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (free_fall/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from free_fall import data, strategy as st
ret = data.fetch_pair()                        # cache-first
c = st.carry_vs_crash(ret["SVXY"])


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | SVXY +11.8%/yr; post-2018 +10%/yr |
| Tradability | **Fragile** | skew −4.8, −95% DD, −83% day |
| Naive hold survives? | **Busted** | 1× inverse-VIX ETPs liquidated 2018 |

> 💡 *In plain words:* real pennies, real steamroller.

## 1 · The claim, steelmanned

- **H₁:** short-vol earns a positive carry (the variance risk premium).
- **H₂:** the left tail is survivable for a buy-and-hold.
- **H₃:** the premium persists after the crash.

## 2 · So what? — what rides on each

H₁ is the premium; H₂ decides whether you can hold it; H₃ whether it's still there. A real premium you can't hold is a trap.

## 3 · How we'd know — the protocol

Return/Sharpe/skew/drawdown vs SPY → worst day → carry-vs-crash decomposition → post-Volmageddon sub-period.

## 4 · The teardown

### 4.1 The carry and the tail

In [2]:
import pandas as pd
display(pd.DataFrame({'SVXY':st.summary(ret['SVXY']),'SPY':st.summary(ret['SPY'])}).T[['cagr','sharpe','vol_ann','max_drawdown','skew','worst_day']].round(3))

,cagr,sharpe,vol_ann,max_drawdown,skew,worst_day
SVXY,0.117,0.559,0.553,-0.952,-4.754,-0.830
SPY,0.157,0.954,0.168,-0.337,-0.287,-0.109


> 💡 *In plain words:* +11.8%/yr (**H₁ holds**) but skew −4.8 and −95% drawdown (**H₂ rejected**).

### 4.2 Carry vs crash

In [3]:
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in c.items()})
wd,wdt=st.worst_day(ret['SVXY']); print(f'worst day {wd:+.0%} on {wdt.date()}')

{'median_day_bp': 40.2248, 'mean_day_bp': 12.272, 'worst_day': -0.8296, 'n_crash_days': 5, 'crash_days_total': -0.948}
worst day -83% on 2018-02-06


> 💡 *In plain words:* +40 bp on a normal day; five crash days take −95%. The premium is rent on a catastrophe.

### 4.3 The premium persists

In [4]:
post=ret.loc['2018-05-01':]; s=st.summary(post['SVXY'])
print(f"post-Volmageddon (de-levered -0.5x): CAGR {s['cagr']:+.1%}, Sharpe {s['sharpe']:.2f}, maxDD {s['max_drawdown']:.0%}")

post-Volmageddon (de-levered -0.5x): CAGR +10.1%, Sharpe 0.45, maxDD -62%


> 💡 *In plain words:* +10%/yr resumes — **H₃ holds**. The premium didn't die; the 2018 holders did. The tail is the whole story.

## 5 · The verdict

H₁/H₃ hold, H₂ rejected → Signal `REAL`, Tradability `FRAGILE`, naive hold `BUSTED`.

## 6 · Could you trade it?

Sized small with an explicit tail hedge (long deep OTM puts / VIX calls), like insurance underwriting — never as a buy-and-hold yield product. The premium is the wage; the −83% day is the job.

## 7 · Going further

Forks: (a) a vol-targeted / tail-hedged short-vol overlay (does managing the tail rescue the Sharpe?); (b) the variance-swap carry vs the ETF; (c) short-vol blended into a 60/40 as a small sleeve. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).